# Caps and bidding — how the optimal bid and delivery rate respond to the penalty cap

Companion to [`../bidding/bidding.ipynb`](../../bidding/bidding.ipynb). Same primitives: a
solver bids $s$, wins the auction iff $s \ge s_{\text{ref}}$ (the competitor's reference bid),
and on a win draws a routing surplus $\sigma$ from a distribution ($\sigma = 0$ means the
routing failed). Conditional on winning:

- **reward** (paid on delivery): $R = \min(c_u,\, s - s_{\text{ref}})$
- **penalty** (paid on default): $P = \min(c_l,\, s_{\text{ref}})$
- deliver payoff $R + \sigma - s$, default payoff $-P$.

In this repo's vocabulary $c_l \leftrightarrow$ `penalty_cap`, $c_u \leftrightarrow$ `reward_cap_upper`.

Three solver regimes, in increasing order of information at the deliver/default decision:

| regime | delivers iff |
|---|---|
| **naive** (commit) | $\sigma > 0$ — always fulfil unless the routing outright failed |
| **selective** (commit, pre-planned revert) | $\sigma \ge \bar\sigma(s),\ \ \bar\sigma(s) = \mathbb{E}[\sigma^*(s, s_{\text{ref}}) \mid \text{win}]$ |
| **defer** (case 2) | $\sigma \ge \sigma^*(s, s_{\text{ref}}) = s - R - P$ per realised $s_{\text{ref}}$ |

**This notebook's question.** For each penalty cap $c_l$ the solver re-optimises its bid
$s^*(c_l) = \arg\max_s \Pi(s)$. We sweep $c_l$ along the x-axis and track two comparative statics:

- **left panel:** the optimal bid $s^*(c_l)$,
- **right panel:** the **delivery rate** at that bid — $P(\text{deliver} \mid \text{win}) = 1 - \text{revert rate}$.

Sliders control the routing ($\mu, \sigma_{\text{std}}, p_{\text{fail}}, n_{\text{bins}}$) and the
**reward** cap $c_u$ (held fixed while $c_l$ sweeps).

*What to expect:* naive always delivers when the routing works, so its delivery rate is flat at
the ceiling $q = 1 - p_{\text{fail}}$. selective and defer walk away from bad realisations when
the penalty is cheap, so their delivery rate starts lower and **rises** toward the ceiling as
$c_l$ grows and voluntary default stops paying.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [ ]:
# ----------------------------------------------------------------------
# Model.  Python port of bidding.ipynb (verified to match the Julia
# implementation to machine precision).  A `routing` is a pair of arrays
# (sig, p); sig = 0 marks a routing failure (forced default).
# ----------------------------------------------------------------------
def normalize(sig, p):
    sig = np.asarray(sig, float); p = np.asarray(p, float)
    return sig, p / p.sum()

def with_failure(sig, p, p_fail):
    """Mix a positive-surplus routing with mass `p_fail` at sigma = 0."""
    sig = np.asarray(sig, float); p = np.asarray(p, float)
    return normalize(np.concatenate([[0.0], sig]),
                     np.concatenate([[p_fail], (1 - p_fail) * p]))

def discretize_density(density, lo, hi, n):
    edges = np.linspace(lo, hi, n + 1)
    mids = 0.5 * (edges[:-1] + edges[1:])
    w = np.array([density(m) for m in mids], float)
    return normalize(mids, w)

def gaussian_routing(mu, sigstd, lo, hi, n):
    return discretize_density(lambda x: np.exp(-0.5 * ((x - mu) / sigstd) ** 2), lo, hi, n)


def _partial_moments(sig, p):
    """Build q_and_mu(tau) for arrays tau:
        q(tau)  = P(sigma > 0 and sigma >= tau)
        mu(tau) = E[sigma * 1{sigma > 0 and sigma >= tau}]
    via suffix sums over the positive-surplus atoms.  Returns also q_total."""
    pos = sig > 0
    s = sig[pos]; pp = p[pos]
    order = np.argsort(s)
    s_sorted = s[order]; pp_sorted = pp[order]
    suf_p = np.concatenate([np.cumsum(pp_sorted[::-1])[::-1], [0.0]])
    suf_mu = np.concatenate([np.cumsum((pp_sorted * s_sorted)[::-1])[::-1], [0.0]])

    def q_and_mu(tau):
        idx = np.searchsorted(s_sorted, tau, side="left")  # count of atoms < tau
        return suf_p[idx], suf_mu[idx]

    return q_and_mu, float(suf_p[0])


def model_curves(routing, S, S_ref, c_u, c_l):
    """Expected payoff Pi(s) over the bid grid S and delivery rate
    (conditional on winning) over S, for all three regimes.

    Pi(s) = (1/|S_ref|) * sum_{s_ref} 1{win} * E_sigma[payoff]   (losing -> 0)."""
    sig, p = routing
    q_and_mu, q_tot = _partial_moments(sig, p)

    Sm = S[:, None]; Sr = S_ref[None, :]
    win = Sm >= Sr                                  # (ns, nr)
    R = np.minimum(c_u, Sm - Sr)                    # reward, valid where win
    P = np.minimum(c_l, S_ref)[None, :]             # penalty, by s_ref
    nwin = win.sum(1); nwin_safe = np.maximum(nwin, 1)

    def avg_over_win(g):
        return np.where(win, g, 0.0).sum(1) / len(S_ref)

    # naive: deliver iff sigma > 0  (threshold tau = 0)
    mu_pos = q_and_mu(np.zeros(1))[1][0]
    g_naive = q_tot * (R - Sm) + mu_pos - (1 - q_tot) * P
    Pi_naive = avg_over_win(g_naive)
    deliv_naive = np.full(len(S), q_tot)            # constant in s

    # selective: deliver iff sigma >= sigma_bar(s), a deterministic threshold per s
    sigstar = Sm - R - P
    sbar = np.where(win, sigstar, 0.0).sum(1) / nwin_safe
    sbar = np.where(nwin > 0, sbar, 0.0)
    q_sel, mu_sel = q_and_mu(sbar)
    g_sel = (q_sel[:, None] * R + mu_sel[:, None]
             - q_sel[:, None] * Sm - (1 - q_sel)[:, None] * P)
    Pi_sel = avg_over_win(g_sel)
    deliv_sel = q_sel                               # same across winning s_ref

    # defer: deliver iff sigma >= sigma*(s, s_ref), a per-realisation threshold
    tau = Sm - R - P
    q_def, mu_def = q_and_mu(tau)
    g_def = q_def * R + mu_def - q_def * Sm - (1 - q_def) * P
    Pi_def = avg_over_win(g_def)
    deliv_def = np.where(win, q_def, 0.0).sum(1) / nwin_safe

    return dict(
        Pi=dict(naive=Pi_naive, selective=Pi_sel, defer=Pi_def),
        deliv=dict(naive=deliv_naive, selective=deliv_sel, defer=deliv_def),
        win_prob=nwin / len(S_ref),
    )


def optima_for_cap(routing, S, S_ref, c_u, c_l):
    """argmax bid s* and the delivery rate / win prob at s*, per regime."""
    cur = model_curves(routing, S, S_ref, c_u, c_l)
    out = {}
    for reg in ("naive", "selective", "defer"):
        i = int(np.argmax(cur["Pi"][reg]))
        out[reg] = dict(s_star=float(S[i]), deliv=float(cur["deliv"][reg][i]),
                        win=float(cur["win_prob"][i]))
    return out


def cap_sweep(routing, S, S_ref, c_u, c_l_grid):
    """Optimal bid, delivery rate and win prob for each penalty cap in the grid."""
    res = {reg: dict(s_star=[], deliv=[], win=[]) for reg in ("naive", "selective", "defer")}
    for c_l in c_l_grid:
        o = optima_for_cap(routing, S, S_ref, c_u, c_l)
        for reg in res:
            for k in res[reg]:
                res[reg][k].append(o[reg][k])
    return {reg: {k: np.asarray(v) for k, v in d.items()} for reg, d in res.items()}

In [ ]:
# Sanity checks: delivery rates live in [0, 1], naive is flat at q = 1 - p_fail,
# and selective / defer rise monotonically toward that ceiling as c_l grows.
def _selfcheck():
    S = np.arange(0, 201, 1.0); S_ref = np.arange(0, 201, 2.0)
    r = with_failure(*gaussian_routing(120, 30, 0, 200, 24), 0.15)
    q = float(r[1][r[0] > 0].sum())
    sw = cap_sweep(r, S, S_ref, 200.0, np.linspace(0, 250, 51))
    for reg in ("naive", "selective", "defer"):
        d = sw[reg]["deliv"]
        assert (d >= -1e-9).all() and (d <= 1 + 1e-9).all(), f"{reg} delivery outside [0,1]"
        assert (np.diff(d) >= -1e-9).all(), f"{reg} delivery not monotone in c_l"
        assert abs(d[-1] - q) < 1e-6, f"{reg} does not converge to ceiling q"
    assert np.allclose(sw["naive"]["deliv"], q), "naive delivery should be flat at q"
    print(f"self-check OK  (ceiling q = 1 - p_fail = {q:.3f})")

_selfcheck()

In [ ]:
# ----------------------------------------------------------------------
# Interactive figure: optimal bid s*(c_l) and delivery rate at s* as the
# penalty cap c_l sweeps the x-axis.  Sliders set the routing and the
# (fixed) reward cap c_u.  Colours match bidding.ipynb.
# ----------------------------------------------------------------------
S       = np.arange(0, 201, 1.0)     # bid grid
S_ref   = np.arange(0, 201, 2.0)     # competitor reference grid (uniform)
N_CAPS  = 101                         # resolution of the c_l sweep
REGS    = [("naive", "black"), ("selective", "tomato"), ("defer", "dodgerblue")]

plt.ioff()
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)
plt.ion()
fig.canvas.header_visible = False

linesL = {reg: axL.plot([], [], color=c, lw=2, label=reg)[0] for reg, c in REGS}
linesR = {reg: axR.plot([], [], color=c, lw=2, label=reg)[0] for reg, c in REGS}
ceil_line = axR.axhline(0.0, ls="--", lw=1, color="gray", alpha=0.8)

axL.set(xlabel="penalty cap  $c_l$", ylabel="optimal bid  $s^*$", title="Optimal bid")
axR.set(xlabel="penalty cap  $c_l$", ylabel="delivery rate   $P(\\mathrm{deliver}\\mid\\mathrm{win})$",
        title="Delivery rate at $s^*$", ylim=(-0.02, 1.04))

sl = dict(
    mu      = widgets.FloatSlider(value=120, min=0,    max=300, step=1,    description="μ"),
    sigstd  = widgets.FloatSlider(value=30,  min=1,    max=100, step=1,    description="σ_std"),
    p_fail  = widgets.FloatSlider(value=0.15,min=0,    max=1,   step=0.01, description="p_fail",
                                  readout_format=".2f"),
    n_bins  = widgets.IntSlider  (value=24,  min=4,    max=64,  step=1,    description="n_bins"),
    c_u     = widgets.FloatSlider(value=200, min=0,    max=500, step=5,    description="c_u (reward)"),
    c_l_max = widgets.FloatSlider(value=250, min=20,   max=500, step=10,   description="c_l max"),
)

def update(_=None):
    v = {k: w.value for k, w in sl.items()}
    routing = with_failure(*gaussian_routing(v["mu"], v["sigstd"], 0, 200, int(v["n_bins"])),
                           v["p_fail"])
    c_l_grid = np.linspace(0, v["c_l_max"], N_CAPS)
    sw = cap_sweep(routing, S, S_ref, v["c_u"], c_l_grid)

    smax = 1.0
    for reg, _c in REGS:
        linesL[reg].set_data(c_l_grid, sw[reg]["s_star"])
        linesR[reg].set_data(c_l_grid, sw[reg]["deliv"])
        smax = max(smax, float(sw[reg]["s_star"].max()))

    q = 1 - v["p_fail"]
    ceil_line.set_ydata([q, q])
    ceil_line.set_label(f"ceiling  $1-p_{{fail}}$ = {q:.2f}")

    axL.set_xlim(0, v["c_l_max"]); axR.set_xlim(0, v["c_l_max"])
    axL.set_ylim(0, smax * 1.08)
    axL.legend(loc="best", fontsize=8); axR.legend(loc="best", fontsize=8)
    fig.suptitle(f"μ={v['mu']:.0f}   σ_std={v['sigstd']:.0f}   p_fail={v['p_fail']:.2f}   "
                 f"n_bins={int(v['n_bins'])}   reward cap c_u={v['c_u']:.0f}", fontsize=9)
    fig.canvas.draw_idle()

for w in sl.values():
    w.observe(update, names="value")
update()

display(widgets.HBox([widgets.VBox(list(sl.values())), fig.canvas]))

### Reading the figure

- **Delivery rate (right).** *naive* sits exactly on the dashed ceiling $1 - p_{\text{fail}}$: it
  never walks away voluntarily, so it delivers on every non-failed routing regardless of $c_l$.
  *selective* and *defer* start below the ceiling — when the penalty is cheap they default on poor
  realisations — and climb to meet it as $c_l$ rises and voluntary default stops paying. *defer*,
  which decides per realised $s_{\text{ref}}$, sits below *selective* at low $c_l$ because it can
  be even more opportunistic about walking away.
- **Optimal bid (left).** At low $c_l$, *defer* (and to a lesser extent *selective*) bid **above**
  *naive*: a cheap walk-away option lowers the effective "must-deliver" probability, so an extra
  unit of bid is cheaper at the margin and the solver bids more aggressively to win. As $c_l$ grows
  the option loses value and all three bids converge.
- The two panels move together: an aggressive bid bought with a cheap default option (high $s^*$,
  low delivery) is exactly what a small penalty cap permits. Raising $c_l$ trades auction
  aggression for fulfilment.

`cap_sweep` also returns `win` (auction win probability $P(s^* \ge s_{\text{ref}})$) per regime if
you want to plot it alongside the delivery rate.